In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State 
from dash import callback, ctx # Added callback and ctx for filtering aid
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# initialized CRUD Python module file with correct name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Update credentials with username and password and CRUD Python module name
username = "aacuser"
password = "aacuser2026"
host = 'localhost' 
port = 27017 
database = 'aac' 
collection = 'animals'

# Connect to database via CRUD Module
db = AnimalShelter(username, password, host, port, database, collection)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # The Grazioso Salvare logo
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    # Header containing Grazioso Salvare’s logo
    html.Div([
      html.A([
        html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style = {
                    'width': '250px',
                    'background-color' : '#fcfcfc' # White
                })],
        href = 'https://www.snhu.edu',)
    ],
    style = {
        'text-align' : 'center',
        'background-color' : '#c9134b', # Red
        'padding' : '10px'
    }),
    # Unique identifier
    html.Center(html.B(html.H1('Fetch&Rescue - Drilling Server Authentication'))),
    html.Hr(),
    # Buttons that control the interactive filtering options
    html.Div(className='buttonRow', 
            style={'display' : 'flex', 'gap' : '10px'},
                children=[
                    html.Button(id='filter-button-water', n_clicks=0, children='Water Rescue'),
                    html.Button(id='filter-button-mountain', n_clicks=0, children='Mountain Rescue'),
                    html.Button(id='filter-button-disaster', n_clicks=0, children='Disaster Rescue'),
                    html.Button(id='filter-button-reset', n_clicks=0, children='Reset')
                ]),
    html.Hr(),
    # Table containing the MongoDB data
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        style_table = {
            # Create horizontal scrolling on table
            # Ref: https://community.plotly.com/t/dash-table-experiments-scrollbar-disappeared-in-chrome/15211/6
            'overflowX' : 'scroll',
            'width' : '100%'            
        },
        # Features for the interactive data table to make it user-friendly for clients
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable="single",
        row_selectable="single",
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0], # Default row when map is initalized
        page_action="native",
        page_current=0,
        page_size=10
    ),
    html.Br(),
    html.Hr(),
    #This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={
             'display' : 'flex',
             'flexWrap' : 'wrap' # Allows stacking on narrow windows
         },
             children=[
        html.Div(
            id='map-id',
            className='col s12 m6',
            style = {
                # Ref: https://stackoverflow.com/questions/37386244/what-does-flex-1-mean
                'flex' : '1 1 48%', # dynamically use 48% of the width
                'minWidth' : '350px',
                'padding' : '10px' 
            }
            ),
        html.Div(
            id='graph-id',
            className='col s12 m6',
            style = {
                # Ref: https://stackoverflow.com/questions/37386244/what-does-flex-1-mean
                'flex' : '1 1 48%', # dynamically use 48% of the width
                'minWidth' : '350px',
                'padding' : '10px'
            }
            )
        ]),
    html.Br(),
    # Footer
    html.Footer([
        html.Div([
            html.P("©2026 Grazioso Salvare - Fetch & Rescue Dashboard"),
            html.P("Developed by: Anyka Perzynski-Drilling / June 2026 - All rights reserved")
        ])
    ], style = {
        'text-align' : 'center',
        'color' : 'white',
        'background-color' : '#c9134b', # Red
        'height' : '100%',
        'padding' : '10px',
        'margin-top' : '20px'
    })
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(Output('datatable-id','data'),
              [Input('filter-button-water', 'n_clicks'),
               Input('filter-button-mountain', 'n_clicks'),
               Input('filter-button-disaster', 'n_clicks'),
               Input('filter-button-reset', 'n_clicks')
              ])

# Ref: https://dash.plotly.com/advanced-callbacks
def update_dashboard(fltr_water, fltr_mountain, fltr_disaster, fltr_reset):
    ## Queries - as specified by Grazioso Salvare
    water_query = {
        "animal_type":"Dog",
        "breed" : {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]}, 
        "sex_upon_outcome": "Intact Female", 
        "age_upon_outcome_in_weeks" : {"$gte": 26, "$lte": 156}
    }
    mountain_query = {
        "animal_type":"Dog",
        "breed" : {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]}, 
        "sex_upon_outcome": "Intact Male", 
        "age_upon_outcome_in_weeks" : {"$gte": 26, "$lte": 156}
    }
    disaster_query = {
        "animal_type":"Dog",
        "breed" : {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]}, 
        "sex_upon_outcome": "Intact Male", 
        "age_upon_outcome_in_weeks" : {"$gte": 20, "$lte": 300}
    }
    
    # Default case
    button_id = 'filter-button-reset'
    
    # Check what triggered the callback
    if not ctx.triggered_id:
        # Default case
        df = pd.DataFrame.from_records(db.read({}))
    else:
        button_id = ctx.triggered_id
    
    # If button was triggered, update the view by the appropriate query
    if (button_id == 'filter-button-water'):
        df = pd.DataFrame.from_records(db.read(water_query))
    elif (button_id == 'filter-button-mountain'):
        df = pd.DataFrame.from_records(db.read(mountain_query))
    elif (button_id == 'filter-button-disaster'):
        df = pd.DataFrame.from_records(db.read(disaster_query))
    # Default case show all
    else:
         df = pd.DataFrame.from_records(db.read({}))
    
    # Cleanup Mongo _id field
    df.drop(columns=['_id'],inplace=True)
    return df.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
# Ref: https://plotly.com/python/bar-charts/
def update_graphs(viewData):
    # If no data is available, return blank
    if viewData is None:
        return []
    
    # Convert dictionary to dataframe
    dff = pd.DataFrame.from_dict(viewData)
    
    # Data
    # Ref: https://www.geeksforgeeks.org/data-analysis/how-to-name-the-column-when-using-valuecounts-function-in-pandas/
    # Note: .head(10) is used to make the entire data set legible
    breed_data = dff['breed'].value_counts().head(10).reset_index()
    breed_data.columns = ['Breed', 'Count']
    
    fig = px.bar(breed_data, x='Breed', y='Count', title='Top 10 Preferred Animal Breeds', height=500)
    
    # Autoscale with margins
    # Ref: https://plotly.com/python/setting-graph-size/
    fig.update_layout(autosize=True, margin=dict(l=40, r=40, t=40, b=40))
    
    return [
        dcc.Graph(
            figure = fig,
            style={
                # Match wrapper elements declared above
                'width' : '100%',
                'height' : '100%'
            },
            config={'responsive' : True}
        )    
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
    
    # Index range check
    # Prevent "Index Out of Range" error:
    # When a row is selected and then a filter is applied to the chart,
    # the indexes on the table will change.
    
    # If the index previously selected row is greater than or equal to
    # total number of rows in the new dataset, index = 0
    if row >= len(dff):
        row = 0
    
    # Check in case the filtered dataset is empty
    if dff.empty:
        return []
    
    # Try to assign data values from the filtered dataset
    # If the dataset suddenly has fewer columns than expected,
    # return empty
    try:
        lat = dff.iloc[row,13]
        lon = dff.iloc[row,14]
        breed = dff.iloc[row,4]
        name = dff.iloc[row,9]
    except IndexError as e:
        print(f"Mapping Error: Missing expected column {e}.")
        return []
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '100%', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[lat,lon], children=[
                dl.Tooltip(breed),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(name)
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(mode='external', port=8052)
#app.run_server(debug=True)

Dash app running on https://pianowilliam-meternissan-3000.codio.io/proxy/8052/
